In [ ]:
import re
import curl_cffi.requests as requests
from parsel import Selector
import json

### Letterboxd Scraping (handling AJAX requests for `/stats` & `/rating-histogram`)

In [ ]:
# Initialize session with Chrome TLS/HTTP2 fingerprint
session = requests.Session(impersonate="chrome124")

BASE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

def get_letterboxd_data(tmdb_id: int) -> dict:
    """
    Fetches main page, rating histogram, and stats for a given Letterboxd film.
    """
    
    tmdb_url = f"https://letterboxd.com/tmdb/{tmdb_id}/"
    
    # 1. Fetch main HTML page for JSON-LD
    r = session.get(tmdb_url, headers=BASE_HEADERS)
    r.raise_for_status()
    main_url = r.url
    slug = re.search(r'/film/([^/]+)/?', main_url).group(1)
    sel = Selector(text=r.text)
    
    # Extract JSON-LD block
    ld_text = sel.css('script[type="application/ld+json"]::text').get()
    ld_text = ld_text.replace('/* <![CDATA[ */', '').replace('/* ]]> */', '').strip()
    ld_data = json.loads(ld_text)
    
    agg = ld_data.get("aggregateRating", {})
    total_rating_count = agg.get("ratingCount", 0)
    
    result = {
        "slug": slug,
        "title": ld_data.get("name"),
        "year": ld_data.get("dateCreated", "")[:4],
        "ratingValue": agg.get("ratingValue"),
        "ratingCount": total_rating_count,
        "reviewCount": agg.get("reviewCount"),
        "histogram": [],
        "stats": {}
    }

    # 2. Fetch Rating Histogram CSI endpoint
    csi_headers = {
        **BASE_HEADERS,
        "Accept": "*/*",
        "Referer": main_url,
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
    }
    
    r_hist = session.get(
        f"https://letterboxd.com/csi/film/{slug}/rating-histogram/", 
        headers=csi_headers
    )
    if r_hist.status_code == 200:
        sel_hist = Selector(text=r_hist.text)
        
        # Parse each row in the histogram table
        for col in sel_hist.css('tr.column'):
            label = col.css('th._sr-only::text').get(default='').strip()
            title_attr = col.css('a.barcolumn::attr(title)').get(default='')
            style_attr = col.css('::attr(style)').get(default='')
            
            # 1. Extract raw integer count (e.g., "602" from "602 half-★ ratings (0%)")
            count_match = re.search(r'([\d,]+)', title_attr)
            count = int(count_match.group(1).replace(',', '')) if count_match else 0
            
            # 2. Extract the CSS --value (relative chart height, max is 1.0)
            value_match = re.search(r'--value:\s*([\d.]+)', style_attr)
            bar_value = float(value_match.group(1)) if value_match else 0.0
            
            # 3. Calculate the true exact percentage based on total ratings
            exact_pct = (count / total_rating_count * 100) if total_rating_count > 0 else 0.0
            
            result["histogram"].append({
                "rating": label,
                "count": count,
                "bar_value": bar_value,            # The CSS --value float
                "percentage_exact": round(exact_pct, 6) # True float percentage of total votes
            })

    # 3. Fetch Stats CSI endpoint
    r_stats = session.get(
        f"https://letterboxd.com/csi/film/{slug}/stats/", 
        headers=csi_headers
    )
    if r_stats.status_code == 200:
        sel_stats = Selector(text=r_stats.text)
        
        def parse_aria_label(text):
            if not text:
                return None
            text = text.replace('&nbsp;', ' ')
            match = re.search(r'([\d,]+)', text)
            return int(match.group(1).replace(',', '')) if match else None
            
        result["stats"] = {
            "watches": parse_aria_label(sel_stats.css('div.-watches::attr(aria-label)').get()),
            "lists": parse_aria_label(sel_stats.css('div.-lists::attr(aria-label)').get()),
            "likes": parse_aria_label(sel_stats.css('div.-likes::attr(aria-label)').get()),
            "top500": parse_aria_label(sel_stats.css('div.-topFilms::attr(aria-label)').get())
        }

    return result

In [ ]:
# Test with "Mirror" (TMDB ID: 1396)
data = get_letterboxd_data(tmdb_id=1396)
print(json.dumps(data, indent=2, ensure_ascii=False))

In [ ]:
counts = [i['count'] for i in data['histogram']]
stars = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]

accum = 0
for i in range(10):
    accum += (counts[i] * stars[i])

accum / sum(counts)

In [ ]:
# Test with "I Saw the Devil" (TMDB ID: 49797)
data = get_letterboxd_data(tmdb_id=49797)
print(json.dumps(data, indent=2, ensure_ascii=False))

In [ ]:
counts = [i['count'] for i in data['histogram']]
stars = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]

accum = 0
for i in range(10):
    accum += (counts[i] * stars[i])

accum / sum(counts)

### Rotten Tomatoes Scraping

In [ ]:
session = requests.Session(impersonate="chrome124")

BASE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

def get_rt_data(url: str) -> dict:
    """
    Fetches Rotten Tomatoes scorecard data for a given movie slug.
    e.g., slug = "backrooms"
    """
    r = session.get(url, headers=BASE_HEADERS)
    r.raise_for_status()
    
    sel = Selector(text=r.text)
    
    # Helper to extract numbers from text strings
    def extract_int(text):
        if not text:
            return 0
        match = re.search(r'([\d,]+)', text)
        return int(match.group(1).replace(',', '')) if match else 0

    # 1. Critics Score (Tomatometer)
    critics_score_text = sel.css('rt-text[slot="critics-score"]::text').get(default='').strip()
    critics_score = int(critics_score_text.replace('%', '')) if critics_score_text else None
    
    # 2. Critics Reviews Count
    critics_reviews_text = sel.css('rt-link[slot="critics-reviews"]::text').get(default='').strip()
    critics_reviews = extract_int(critics_reviews_text)
    
    # 3. Audience Score (Popcornmeter)
    audience_score_text = sel.css('rt-text[slot="audience-score"]::text').get(default='').strip()
    audience_score = int(audience_score_text.replace('%', '')) if audience_score_text else None
    
    # 4. Audience Reviews Count
    audience_reviews_text = sel.css('rt-link[slot="audience-reviews"]::text').get(default='').strip()
    audience_reviews = extract_int(audience_reviews_text)

    return {
        "url": url,
        "critics_score": critics_score,
        "critics_reviews": critics_reviews,
        "audience_score": audience_score,
        "audience_reviews": audience_reviews
    }

In [ ]:
url = "https://www.rottentomatoes.com/m/i_saw_the_devil_2010"
data = get_rt_data(url=url)
print(json.dumps(data, indent=2))

### IMDb Scraping

In [ ]:
from urllib.parse import quote

ENDPOINT = "https://caching.graphql.imdb.com/"
OPERATION = "Title_Storyline"

SHA256 = "bbf29ee4ceeefcf2d0825e0e57e3821aa2e11166b7cf820e1b40fb21095d7b08"

HEADERS = {
    "accept": "application/graphql+json, application/json",
    "content-type": "application/json",
    "referer": "https://www.imdb.com/",
    "origin": "https://www.imdb.com",
    "x-imdb-client-name": "imdb-web-next-localized",
    "x-imdb-user-country": "US",
    "x-imdb-user-language": "en-US",
}

def fetch_title(session, imdb_id):
    variables = {
        "isInMachineTranslateWeblab": True,
        "locale": "en-US",
        "titleId": imdb_id,
    }
    extensions = {"persistedQuery": {"sha256Hash": SHA256, "version": 1}}
    url = (
        f"{ENDPOINT}?operationName={OPERATION}"
        f"&variables={quote(json.dumps(variables, separators=(',', ':')))}"
        f"&extensions={quote(json.dumps(extensions, separators=(',', ':')))}"
    )
    r = session.get(url, headers=HEADERS, impersonate="chrome", timeout=25)
    return r.json( )

In [ ]:
session = requests.Session()
fetch_title(session, "tt0021079")

In [ ]:
# --- copied from imdb_fetch.py (the bits route 2 depends on) ---
ENDPOINT = "https://caching.graphql.imdb.com/"

HEADERS = {
    "accept": "application/graphql+json, application/json",
    "content-type": "application/json",
    "referer": "https://www.imdb.com/",
    "origin": "https://www.imdb.com",
    "x-imdb-client-name": "imdb-web-next-localized",
    "x-imdb-user-country": "US",
    "x-imdb-user-language": "en-US",
}

# --- route 2: inline (non-persisted) query for the ratings histogram ---
QUERY = """
query TitleRatingsHistogram($id: ID!) {
  title(id: $id) {
    id
    ratingsSummary { aggregateRating voteCount }
    aggregateRatingsBreakdown {
      histogram { histogramValues { rating voteCount } }
    }
  }
}
""".strip()


def fetch_ratings_histogram(session, imdb_id):
    body = {
        "operationName": "TitleRatingsHistogram",
        "query": QUERY,
        "variables": {"id": imdb_id},
    }
    r = session.post(
        ENDPOINT,
        headers=HEADERS,
        data=json.dumps(body),
        impersonate="chrome",
        timeout=25,
    )
    r.raise_for_status()
    return r.json()

In [ ]:
session = requests.Session()
data = fetch_ratings_histogram(session, "tt0021079")
print(json.dumps(data, indent=2, ensure_ascii=False))